# Day 5 提升模型对比

Day 5 的实际重点是：在 Day 4 baseline 基础上训练第一版 Random Forest / XGBoost，比较基础缺失处理策略和类别不平衡处理效果。Day 5 不做 GridSearch、不做阈值遍历，也不做风险分层。

## 1. 导入依赖并读取 cfg

路径、标签映射、缺失值 token、业务成本、默认阈值和模型参数都从 `config/config.yaml` 读取。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.models.train_advanced import train_and_evaluate_advanced_models

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
cfg.default_threshold, cfg.high_missing_threshold, cfg.advanced_models

(0.5,
 0.8,
 {'random_forest': {'n_estimators': 200,
   'max_depth': None,
   'min_samples_leaf': 1,
   'class_weight': 'balanced',
   'n_jobs': -1},
  'xgboost': {'n_estimators': 300,
   'max_depth': 4,
   'learning_rate': 0.05,
   'subsample': 0.8,
   'colsample_bytree': 0.8,
   'eval_metric': 'logloss',
   'n_jobs': -1}})

## 2. 读取 train/test 并映射 target

继续使用官方 train/test 划分，不合并后重新切分。

In [2]:
train_df, test_df = load_train_test_with_target(cfg)

pd.DataFrame(
    [
        {"dataset": "train", "rows": train_df.shape[0], "columns": train_df.shape[1]},
        {"dataset": "test", "rows": test_df.shape[0], "columns": test_df.shape[1]},
    ]
)

,dataset,rows,columns
0,train,60000,172
1,test,16000,172


## 3. 读取 Day 4 baseline 指标

Day 5 的模型结果会和 Day 4 Logistic Regression baseline 对比。

In [3]:
baseline_metrics_path = cfg.metrics_dir / "day4_baseline_metrics.csv"
baseline_metrics = pd.read_csv(baseline_metrics_path)

logistic_baseline = baseline_metrics[
    baseline_metrics["model_name"].eq("logistic_regression_balanced")
].copy()

logistic_baseline[
    ["model_name", "strategy", "precision", "recall", "f2", "average_precision", "fn", "total_cost"]
]

,model_name,strategy,precision,recall,f2,average_precision,fn,total_cost
1,logistic_regression_balanced,median_all,0.481894,0.922667,0.779982,0.798196,29,18220
3,logistic_regression_balanced,drop_high_missing_median,0.486034,0.928000,0.785199,0.799356,27,17180


## 4. 定义 Day 5 比较策略

Random Forest 比较三种策略；XGBoost 比较 `median_all` 和 `drop_high_missing_median`。本阶段暂不启用 XGBoost 原生缺失策略，避免 Day 5 范围过大。

In [4]:
strategies = ["median_all", "drop_high_missing_median", "median_with_indicator"]
strategies

['median_all', 'drop_high_missing_median', 'median_with_indicator']

## 5. 训练 Random Forest 和 XGBoost

Random Forest 使用 `class_weight="balanced"`；XGBoost 使用 `scale_pos_weight = neg_count / pos_count`。两者都使用 cfg 中的默认阈值，不做阈值遍历。

In [5]:
advanced_metrics, advanced_predictions = train_and_evaluate_advanced_models(
    train_df=train_df,
    test_df=test_df,
    cfg=cfg,
    strategies=strategies,
)

advanced_metrics

,model_name,strategy,threshold,precision,recall,f1,f2,average_precision,tn,fp,fn,tp,false_positive_cost,false_negative_cost,total_cost,dataset,n_features,n_dropped_features
0,random_forest_balanced,median_all,0.5,0.936709,0.592000,0.725490,0.639033,0.884151,15610,15,153,222,10,500,76650,test,170,0
1,xgboost_scale_pos_weight,median_all,0.5,0.635688,0.912000,0.749179,0.839058,0.911301,15429,196,33,342,10,500,18460,test,170,0
2,random_forest_balanced,drop_high_missing_median,0.5,0.938326,0.568000,0.707641,0.616676,0.883036,15611,14,162,213,10,500,81140,test,168,2
3,xgboost_scale_pos_weight,drop_high_missing_median,0.5,0.627737,0.917333,0.745395,0.839844,0.909098,15421,204,31,344,10,500,17540,test,168,2
4,random_forest_balanced,median_with_indicator,0.5,0.948498,0.589333,0.726974,0.637623,0.890959,15613,12,154,221,10,500,77120,test,339,0


## 6. 与 Day 4 Logistic baseline 对比

核心关注 recall、F2、PR-AUC、FN 和 total cost，不使用 accuracy 作为核心指标。

In [6]:
compare_metrics = pd.concat([logistic_baseline, advanced_metrics], ignore_index=True)

compare_view = compare_metrics[
    [
        "model_name",
        "strategy",
        "precision",
        "recall",
        "f1",
        "f2",
        "average_precision",
        "fp",
        "fn",
        "total_cost",
        "n_features",
        "n_dropped_features",
    ]
].sort_values("total_cost")

compare_view

,model_name,strategy,precision,recall,f1,f2,average_precision,fp,fn,total_cost,n_features,n_dropped_features
1,logistic_regression_balanced,drop_high_missing_median,0.486034,0.928000,0.637947,0.785199,0.799356,368,27,17180,168,2
5,xgboost_scale_pos_weight,drop_high_missing_median,0.627737,0.917333,0.745395,0.839844,0.909098,204,31,17540,168,2
0,logistic_regression_balanced,median_all,0.481894,0.922667,0.633120,0.779982,0.798196,372,29,18220,170,0
3,xgboost_scale_pos_weight,median_all,0.635688,0.912000,0.749179,0.839058,0.911301,196,33,18460,170,0
2,random_forest_balanced,median_all,0.936709,0.592000,0.725490,0.639033,0.884151,15,153,76650,170,0
6,random_forest_balanced,median_with_indicator,0.948498,0.589333,0.726974,0.637623,0.890959,12,154,77120,339,0
4,random_forest_balanced,drop_high_missing_median,0.938326,0.568000,0.707641,0.616676,0.883036,14,162,81140,168,2


## 7. 保存 Day 5 输出

输出文件是本地运行产物，默认不提交 GitHub。

In [7]:
cfg.metrics_dir.mkdir(parents=True, exist_ok=True)
cfg.predictions_dir.mkdir(parents=True, exist_ok=True)

metrics_path = cfg.metrics_dir / "day5_model_compare_metrics.csv"
predictions_path = cfg.predictions_dir / "day5_model_compare_predictions.csv"

advanced_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
advanced_predictions.to_csv(predictions_path, index=False, encoding="utf-8-sig")

metrics_path, predictions_path

(WindowsPath('C:/Scania APS/outputs/metrics/day5_model_compare_metrics.csv'),
 WindowsPath('C:/Scania APS/outputs/predictions/day5_model_compare_predictions.csv'))

## 8. Day 5 小结

- Day 5 已完成第一版 Random Forest / XGBoost 对比结构。
- Random Forest 使用 `class_weight="balanced"` 处理类别不平衡。
- XGBoost 使用 `scale_pos_weight` 处理类别不平衡。
- 本阶段只使用默认阈值 `0.5`，不做阈值成本曲线。
- Day 6 应在固定模型结果基础上做阈值成本分析。